# Agentic_RL_Exec: GSM8K GRPO (Qwen2.5-0.5B, single GPU)

**할 일(사람)**: 런타임을 A100으로 설정 → Run all → **1분 내** Drive 인증 팝업에서 "Allow" 클릭 → 이후 자리 비워도 됨(끝날 때까지 Run All이 블로킹).

이 노트북은 브라우저에서 사람이 Run all로 직접 돌리는 걸 전제로 함(colab-cli 세션 소실 버그 #106 회피 — 상세: vault Colab_Ops-MOC). Drive에 결과 로그를 저장해 세션이 죽어도 산출물이 남게 함.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RUN_DIR = '/content/drive/MyDrive/agentic_rl/runs/' + __import__('datetime').datetime.now().strftime('%Y%m%d_%H%M%S')
os.makedirs(RUN_DIR, exist_ok=True)
print('RUN_DIR =', RUN_DIR)


## Setup: clone + env (idempotent)

In [ ]:
import os, subprocess

if not os.path.exists('/content/verl'):
    subprocess.run(['git', 'clone', '--depth', '1',
                     'https://github.com/verl-project/verl.git', '/content/verl'], check=True)
else:
    print('/content/verl already exists, skip clone')

print(subprocess.run(['git', '-C', '/content/verl', 'log', '--oneline', '-1'],
                      capture_output=True, text=True).stdout)


In [ ]:
import subprocess

# idempotent: skip uv sync if venv already has verl+vllm importable
check = subprocess.run(
    ['bash', '-c',
     'cd /content/verl && uv run --frozen --all-packages --extra vllm --extra fsdp '
     'python3 -c "import verl, vllm, torch; print(\'OK\', torch.cuda.is_available())" 2>&1'],
    capture_output=True, text=True
)
print(check.stdout[-1000:])
need_sync = 'OK True' not in check.stdout

if need_sync:
    print('running uv sync (several GB, may take a while)...')
    r = subprocess.run(['bash', '-c',
        'cd /content/verl && uv sync --frozen --all-packages --extra vllm --extra fsdp'],
        capture_output=True, text=True)
    print(r.stdout[-3000:])
    print('STDERR:', r.stderr[-2000:])
    print('exit', r.returncode)
else:
    print('venv already OK, skip uv sync')


In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout)
r2 = subprocess.run(['bash', '-c',
    'cd /content/verl && uv run --frozen --all-packages --extra vllm --extra fsdp python3 -c '
    '"import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0))"'],
    capture_output=True, text=True)
print(r2.stdout)
print(r2.stderr[-500:])
assert 'True' in r2.stdout, 'CUDA not available, stop here'


## GSM8K preprocessing

In [ ]:
import subprocess, os
if not os.path.exists('/content/data/gsm8k/train.parquet'):
    r = subprocess.run(['bash', '-c',
        'cd /content/verl && uv run --frozen --all-packages --extra vllm --extra fsdp python3 '
        'examples/data_preprocess/gsm8k.py --local_save_dir /content/data/gsm8k'],
        capture_output=True, text=True)
    print(r.stdout[-2000:])
    print(r.stderr[-1000:])
else:
    print('gsm8k parquet already exists, skip')

r = subprocess.run(['ls', '-la', '/content/data/gsm8k/'], capture_output=True, text=True)
print(r.stdout)


## Training: GRPO, Qwen2.5-0.5B-Instruct, single GPU, 10 steps

파이프라인 검증이 목적 — GSM8K strict-mode reward라 짧은 스텝으로 학습 성과는 기대하지 않음(quickstart 샘플도 step0 reward mean 0.004). `MAX_RESPONSE_LENGTH=1024` 유지(짧으면 `####` 형식을 못 맞춰 전원 0점 위험).

`SAVE_FREQ=-1`로 체크포인트 미저장(Drive FUSE 쓰기가 느려서 이번 검증 런엔 불필요) — 로그만 저장.


In [ ]:
import subprocess, os

env = os.environ.copy()
env.update({
    'MODEL_PATH': 'Qwen/Qwen2.5-0.5B-Instruct',
    'TRAIN_FILE': '/content/data/gsm8k/train.parquet',
    'TEST_FILE': '/content/data/gsm8k/test.parquet',
    'NGPUS_PER_NODE': '1',
    'ROLLOUT_TP': '1',
    'ROLLOUT_GPU_MEM_UTIL': '0.5',
    'ROLLOUT_N': '5',
    'TRAIN_BATCH_SIZE': '20',
    'PPO_MINI_BATCH_SIZE': '10',
    'PPO_MICRO_BATCH_SIZE_PER_GPU': '2',
    'LOG_PROB_MICRO_BATCH_SIZE_PER_GPU': '2',
    'MAX_PROMPT_LENGTH': '512',
    'MAX_RESPONSE_LENGTH': '1024',
    'ACTOR_LR': '1e-6',
    'PROJECT_NAME': 'agentic_rl_exec',
    'EXPERIMENT_NAME': 'gsm8k_grpo_qwen25_0_5b_pipeline_check',
    'SAVE_FREQ': '-1',
    'TEST_FREQ': '5',
    'TOTAL_EPOCHS': '1',
})

log_path = os.path.join(RUN_DIR, 'train.log')
cmd = (
    'cd /content/verl && bash examples/grpo_trainer/run_qwen3_4b_fsdp.sh '
    'trainer.logger=\'["console"]\' trainer.total_training_steps=10 '
    f'2>&1 | tee {log_path}'
)
print('$', cmd)
r = subprocess.run(['bash', '-c', cmd], env=env)
print('exit', r.returncode)


## Results summary + env snapshot to Drive

In [ ]:
import subprocess, re, os

log_path = os.path.join(RUN_DIR, 'train.log')
with open(log_path) as f:
    log = f.read()

for pat in [r'val/test_score/openai/gsm8k[^\s]*', r'critic/score/mean:[0-9.]+',
            r'^step:\d+.*', ]:
    matches = re.findall(pat, log, re.MULTILINE)
    print(f'--- {pat} ({len(matches)} matches) ---')
    for m in matches[:5]:
        print(m[:300])
    print()

# env snapshot
freeze = subprocess.run(['bash', '-c',
    'cd /content/verl && uv run --frozen --all-packages --extra vllm --extra fsdp pip freeze'],
    capture_output=True, text=True).stdout
with open(os.path.join(RUN_DIR, 'pip_freeze.txt'), 'w') as f:
    f.write(freeze)

smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
with open(os.path.join(RUN_DIR, 'nvidia-smi.txt'), 'w') as f:
    f.write(smi)

print('Saved to', RUN_DIR)
print(subprocess.run(['ls', '-la', RUN_DIR], capture_output=True, text=True).stdout)


## 끝났으면

- `RUN_DIR` 아래 (`train.log`, `pip_freeze.txt`, `nvidia-smi.txt`)를 Drive에서 확인.
- 이 세션은 종료해도 됨 — 결과는 Drive에 남아있음.
- 다음 raw 결과는 로컬 WSL에서 `/mnt/c/.../Google Drive/agentic_rl/runs/...` 경로로 확인 가능(Google Drive Desktop 동기화 시).
